# U-Net r34 on all 707 training images (final model, Charan)

Run 1's recipe (ResNet34 U-Net, 1024 px), trained on train + val of the 7-day split. That's every training image, with nothing held out. There is no honest validation, so there's no early stopping and no checkpoint selection: training runs a fixed 32 epochs (run 1 peaked at epoch 29) and keeps the last epoch. The val PQ printed per epoch is leaky and only shows progress.

Post-processing uses rules tuned on the 7-day split in `filament-ensemble`:
* `full`: run 2's best rule `t_lo .5, peak .8, mean .8, area 400, close 6`
* `full3` = average of full + run 1 + run 2, with ens2's rule `t_lo .5, peak .95, mean .8, area 200, close 6`

**Settings:** GPU T4 x2, Internet on, competition data + kernels `filament-unet-r34`, `filament-unet-r34-week`.

In [ ]:
# 1. Setup
import os, subprocess, sys, json, shutil
from pathlib import Path
import pandas as pd

BRANCH = "charan/ensemble"
REPO = "https://github.com/hsfcoulibaly/aml_project.git"
SRC = Path("/kaggle/working/aml_project")
if not SRC.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(SRC)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "pycocotools"], check=True)
CODE = SRC / "Charan" / "unet"
os.chdir(CODE); sys.path.insert(0, str(CODE))
os.environ["UNET_WORK_DIR"] = "/kaggle/working/unet"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path(os.environ["UNET_WORK_DIR"])
OUT = Path("/kaggle/working/unet_results"); OUT.mkdir(exist_ok=True)
print(subprocess.run(["git", "-C", str(SRC), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Prepare 1024 px inputs (7-day split file; the val images are trained on too)
!python scripts/prepare.py --size 1024 --workers 4 --split splits/week_split.json

In [ ]:
# 3. Train on everything for a fixed 32 epochs (~1.5-2 h)
RUN = "unet_r34_full"
!python scripts/train.py --name {RUN} --train-on-val --encoder resnet34 --epochs 32 --batch 8 --lr 3e-4 --pos-weight 10 --pq-every 4 --workers 4

In [ ]:
h = pd.read_csv(WORK / "runs" / RUN / "history.csv")
display(h.tail(8))

In [ ]:
# 4. Test submissions with the earlier tuned rules
def find_best(slug):
    hits = [p for p in Path("/kaggle/input").rglob("best.pt") if p.parent.name == "unet_results" and slug in p.parts]
    assert len(hits) == 1, (slug, hits)
    return hits[0]
R1, R2 = find_best("filament-unet-r34"), find_best("filament-unet-r34-week")
!python scripts/ensemble.py --model full={WORK}/runs/{RUN}/best.pt --model unet_r1={R1} --model unet_week={R2} \
    --blend full=full:1 --blend full3=full:1,unet_r1:1,unet_week:1 \
    --fixed-rule full=0.5:0.8:0.8:400:6 --fixed-rule full3=0.5:0.95:0.8:200:6 \
    --tta d4 --workers 4 --out-dir {OUT}/ensemble

In [ ]:
# Keep weights, history, sweeps and CSVs; drop the large prepared data
for f in ["best.pt", "history.csv", "args.json"]:
    p = WORK / "runs" / RUN / f
    if p.exists(): shutil.copy(p, OUT / f)
for p in (OUT / "ensemble").iterdir(): print(p.name)
print(json.dumps(json.load(open(OUT / "ensemble" / "summary.json")), indent=1)[:4000])
shutil.rmtree(WORK / "images", ignore_errors=True); shutil.rmtree(WORK / "masks", ignore_errors=True)
(WORK / "runs" / RUN / "last_state.pt").unlink(missing_ok=True)
shutil.rmtree(SRC, ignore_errors=True)